# 212 — Merge Outputs and Resolve Cross-References

Merges per-document CSVs produced by notebook 211 and asks Claude Sonnet to resolve each cross-document reference to a known `section_id`.

| Input | Output |
|---|---|
| `{regulation_id}_sections.csv` (one per doc) | `sections.csv` (merged) |
| `{regulation_id}_requirements.csv` | `requirements.csv` (merged) |
| `{regulation_id}_thresholds.csv` | `thresholds.csv` (merged) |
| `{regulation_id}_references.csv` | `cross_references.csv` (resolved) |

Re-runnable: yes.

In [1]:
import sys, os, json, time
import pandas as pd
import anthropic
from pathlib import Path
from dotenv import load_dotenv

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / '.env')

from src.document.utils import strip_fences, call_claude_stream_json

LAYER2_DIR       = project_root / 'data' / 'layer_2'
INTERMEDIATE_DIR = LAYER2_DIR / 'intermediate'

client     = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
MODEL      = 'claude-sonnet-4-6'
MAX_TOKENS = 8192   # raised from 4096 — 50 refs × 5 fields needs headroom
BATCH_SIZE = 25     # halved to reduce per-call output size

print(f'Intermediate dir: {INTERMEDIATE_DIR}')


Intermediate dir: /Users/emilpastor/Documents/github/loanguard-ai/data/layer_2/intermediate


## Step 1 — Merge per-document CSVs

In [2]:
def load_and_merge(pattern: str) -> pd.DataFrame:
    files = sorted(INTERMEDIATE_DIR.glob(pattern))
    if not files:
        print(f'  No files matching {pattern}')
        return pd.DataFrame()
    dfs = [pd.read_csv(f) for f in files]
    print(f'  Merged {len(files)} file(s) matching {pattern}')
    return pd.concat(dfs, ignore_index=True)


sections_df     = load_and_merge('*_sections.csv')
requirements_df = load_and_merge('*_requirements.csv')
thresholds_df   = load_and_merge('*_thresholds.csv')
references_df   = load_and_merge('*_references.csv')

print(f'\nMerged totals:')
print(f'  sections:     {len(sections_df)}')
print(f'  requirements: {len(requirements_df)}')
print(f'  thresholds:   {len(thresholds_df)}')
print(f'  references:   {len(references_df)}')

# Write merged CSVs to LAYER2_DIR (used by subsequent notebooks)
sections_df.to_csv(LAYER2_DIR / 'sections.csv', index=False)
requirements_df.to_csv(LAYER2_DIR / 'requirements.csv', index=False)
thresholds_df.to_csv(LAYER2_DIR / 'thresholds.csv', index=False)
print('\nMerged CSVs written to data/layer_2/: sections.csv, requirements.csv, thresholds.csv')


  Merged 3 file(s) matching *_sections.csv
  Merged 3 file(s) matching *_requirements.csv
  Merged 3 file(s) matching *_thresholds.csv
  Merged 3 file(s) matching *_references.csv

Merged totals:
  sections:     98
  requirements: 264
  thresholds:   244
  references:   73

Merged CSVs written to data/layer_2/: sections.csv, requirements.csv, thresholds.csv


## Step 2 — Resolve cross-references via Claude

In [3]:
def resolve_batch(refs: list, all_section_ids: list) -> list:
    """Ask Claude to match a batch of references to known section_ids."""
    sections_list = '\n'.join(all_section_ids)
    refs_json     = json.dumps(refs, indent=2)

    system = (
        'You are resolving cross-document references in a regulatory knowledge graph. '
        'Match each reference to the best target section_id from the provided list. '
        'Return ONLY a valid JSON array, no markdown fences, no preamble.'
    )
    user_msg = (
        'Available section_ids:\n'
        + sections_list
        + '\n\nFor each reference below, find the best matching target section_id from the list above.'
        + ' If the referenced document is not in the list, set to_section_id to EXTERNAL.'
        + '\n\nReturn a JSON array where each element has:\n'
        + '  from_section_id   (string): the originating section\n'
        + '  to_section_id     (string): matched section_id, or EXTERNAL\n'
        + '  reference_type    (string): requires | elaborates | supplements | supersedes\n'
        + '  description       (string): brief description of the reference\n'
        + '  confidence        (string): high | medium | low\n'
        + '\nReferences to resolve:\n'
        + refs_json
    )

    return call_claude_stream_json(
        client,
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=system,
        messages=[{'role': 'user', 'content': user_msg}],
    )


all_section_ids = sections_df['section_id'].tolist() if not sections_df.empty else []
refs_list       = references_df.to_dict('records') if not references_df.empty else []

resolved_internal = []
external_refs     = []
batch_times       = []

if refs_list and all_section_ids:
    total_batches  = (len(refs_list) + BATCH_SIZE - 1) // BATCH_SIZE
    pipeline_start = time.perf_counter()

    for i in range(0, len(refs_list), BATCH_SIZE):
        batch     = refs_list[i:i + BATCH_SIZE]
        batch_num = i // BATCH_SIZE + 1
        print(f'Resolving batch {batch_num}/{total_batches}: {len(batch)} references...', end=' ', flush=True)
        t0       = time.perf_counter()
        resolved = resolve_batch(batch, all_section_ids)
        elapsed  = time.perf_counter() - t0
        batch_times.append(elapsed)
        print(f'{elapsed:.1f}s')
        for ref in resolved:
            if ref.get('to_section_id') == 'EXTERNAL':
                external_refs.append(ref)
            else:
                resolved_internal.append(ref)

    total_elapsed = time.perf_counter() - pipeline_start
    print(f'\nTotal API time: {total_elapsed:.1f}s (avg {sum(batch_times) / len(batch_times):.1f}s/batch)')
else:
    print('No references to resolve or no section_ids available.')

print(f'\nResolved: {len(resolved_internal)} internal cross-references')
print(f'External: {len(external_refs)} references to documents outside this set')


Resolving batch 1/3: 25 references... 21.7s
Resolving batch 2/3: 25 references... 28.0s
Resolving batch 3/3: 23 references... 22.2s

Total API time: 71.9s (avg 24.0s/batch)

Resolved: 30 internal cross-references
External: 43 references to documents outside this set


## Step 3 — Write cross_references.csv

In [4]:
COLUMNS = ['from_section_id', 'to_section_id', 'reference_type', 'description', 'confidence']

if resolved_internal:
    cross_df = pd.DataFrame(resolved_internal)

    # Drop any rows where the resolved section_id is not in the known set
    section_id_set = set(all_section_ids)
    valid_from = cross_df['from_section_id'].isin(section_id_set)
    valid_to   = cross_df['to_section_id'].isin(section_id_set)
    invalid    = cross_df[~(valid_from & valid_to)]
    if not invalid.empty:
        print(f'\u26a0  Dropping {len(invalid)} rows with unrecognised section_ids')
    cross_df = cross_df[valid_from & valid_to]

    # Ensure expected columns exist
    for col in COLUMNS:
        if col not in cross_df.columns:
            cross_df[col] = ''
    cross_df[COLUMNS].to_csv(LAYER2_DIR / 'cross_references.csv', index=False)
    print(f'Written cross_references.csv to data/layer_2/: {len(cross_df)} rows')
else:
    pd.DataFrame(columns=COLUMNS).to_csv(LAYER2_DIR / 'cross_references.csv', index=False)
    print('Written empty cross_references.csv to data/layer_2/ (no internal cross-references found)')


Written cross_references.csv to data/layer_2/: 30 rows


## Step 4 — Summary

In [5]:
cross_path = LAYER2_DIR / 'cross_references.csv'
if cross_path.exists():
    cross_df = pd.read_csv(cross_path)

    if not cross_df.empty:
        def doc_prefix(sid: str) -> str:
            parts = str(sid).split('-')
            return '-'.join(parts[:2]) if len(parts) >= 2 else sid

        cross_df['from_doc'] = cross_df['from_section_id'].apply(doc_prefix)
        cross_df['to_doc']   = cross_df['to_section_id'].apply(doc_prefix)

        print('Cross-references by document pair:')
        print(cross_df.groupby(['from_doc', 'to_doc']).size().to_string())

        if 'confidence' in cross_df.columns:
            low_conf = cross_df[cross_df['confidence'] == 'low']
            if not low_conf.empty:
                print(f'\n\u26a0  {len(low_conf)} low-confidence matches — review these:')
                for _, row in low_conf.iterrows():
                    frm  = row['from_section_id']
                    to   = row['to_section_id']
                    desc = row.get('description', '')
                    print(f'  {frm} \u2192 {to}: {desc}')

if external_refs:
    print(f'\nExternal references (not ingested into graph):')
    for ref in external_refs:
        frm  = ref.get('from_section_id', '')
        desc = ref.get('description', '')
        print(f'  {frm}: {desc}')


Cross-references by document pair:
from_doc  to_doc 
APG-223   APS-112     1
          APS-220     3
APS-112   APS-112    21
          APS-220     2
APS-220   APS-112     1
          APS-220     2

External references (not ingested into graph):
  APG-223-S1: This PPG should be read in conjunction with CPS 220 Risk Management.
  APG-223-S1: This PPG should be read in conjunction with CPG 220 Risk Management.
  APG-223-S1: This PPG should be read in conjunction with CPS 510 Governance.
  APG-223-S3: Consistent with CPS 220, where residential mortgage lending is material, the Board and senior management should specifically address it in the risk management framework.
  APG-223-S3: CPS 510 requires the remuneration policy to apply to responsible persons, risk and financial control personnel and all other persons whose activities may affect the financial soundness of the regulated institution.
  APG-223-S3: Refer to CPG 235 for guidance on managing data risk in relation to management inform